## InsightFace vs Original Haar Cascade

**InsightFace Advantages:**
- Multi-face detection (detects all faces in a single pass)
- Automatic face alignment & normalization
- Higher-quality 512D embeddings optimized for recognition
- Built-in face quality scoring
- CUDA GPU acceleration support
- Handles various poses, angles, and lighting conditions

**Original Haar Cascade:**
- Lightweight, CPU-optimized
- Fast for single face detection
- Limited multi-face accuracy
- Requires DeepFace wrapper for embeddings

In [ ]:
# Get geotag history for a specific person
person_id = "photo_bts_face_recog"  # Change to your person ID

geotag_df = pipeline.get_person_geotags(person_id)

if not geotag_df.empty:
    print(f"Geotag history for {person_id}:")
    print(geotag_df)
    print(f"\nLast location:")
    last_location = geotag_df.iloc[-1]
    print(f"  Latitude: {last_location['latitude']}")
    print(f"  Longitude: {last_location['longitude']}")
    print(f"  Timestamp: {last_location['timestamp']}")
else:
    print(f"No geotags found for {person_id}")

## Step 4: View Person's Geotag History
Track where a person was detected across all registered photos

In [ ]:
# Diagnostic: Check Database & Embeddings
print("="*60)
print("DATABASE DIAGNOSTICS (InsightFace)")
print("="*60)
print(f"\nTotal faces registered: {len(pipeline.faces_db)}")
if not pipeline.faces_db.empty:
    print(f"Unique persons: {pipeline.faces_db['person_id'].unique().tolist()}")
    print(f"\nDatabase shape: {pipeline.faces_db.shape}")
    
    # Check for issues
    print(f"\nGeotag Status:")
    print(f"  Latitude - None count: {pipeline.faces_db['latitude'].isna().sum()}")
    print(f"  Longitude - None count: {pipeline.faces_db['longitude'].isna().sum()}")
    
    # Check embeddings
    print(f"\nEmbedding Status:")
    embedding_cols = [f'embedding_{i}' for i in range(512)]
    none_count = 0
    for col in embedding_cols:
        none_count += pipeline.faces_db[col].isna().sum()
    print(f"  Total NaN values in embeddings: {none_count}")
    
    # Sample face
    print(f"\nFirst registered face details:")
    first_face = pipeline.faces_db.iloc[0]
    print(f"  Person ID: {first_face['person_id']}")
    print(f"  Filename: {first_face['filename']}")
    print(f"  Latitude: {first_face['latitude']}")
    print(f"  Longitude: {first_face['longitude']}")
    print(f"  Detection Confidence: {first_face['confidence']:.4f}")
    print(f"  Embedding sample: {[first_face[f'embedding_{i}'] for i in range(5)]}")
    print(f"  Embedding norm: {np.linalg.norm(np.array([first_face[f'embedding_{i}'] for i in range(512)])):.4f}")
else:
    print("Database is empty. Register faces first.")

## Step 3: Database Diagnostics
Check database stats, embeddings quality, and geotag extraction

In [ ]:
# Search for similar faces with detailed debugging
query_image = r"C:\Users\ASUS\Pictures\New folder\OIP.jpg"

if os.path.exists(query_image):
    print(f"Database has {len(pipeline.faces_db)} registered faces")
    print(f"Running search with InsightFace...\n")
    
    # Search with lower threshold since InsightFace embeddings can be highly accurate
    search_results = pipeline.search_similar_faces(query_image, similarity_threshold=0.4)
    
    # Display results
    pipeline.visualize_results(search_results, max_matches=8)
    
    # Debug: Show all similarity scores for query faces
    print("\n" + "="*60)
    print("DEBUG: All Similarity Scores for Each Query Face")
    print("="*60)
    
    query_faces = pipeline._extract_face_embeddings(query_image)
    for q_idx, query_face in enumerate(query_faces):
        query_embedding = query_face['embedding']
        print(f"\nQuery Face {q_idx+1}:")
        
        scores = []
        for db_idx, db_row in pipeline.faces_db.iterrows():
            db_embedding = np.array([db_row[f'embedding_{i}'] for i in range(512)])
            similarity = 1 - cosine(query_embedding, db_embedding)
            scores.append((db_idx, similarity, db_row['person_id']))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        for rank, (db_idx, score, person_id) in enumerate(scores[:8], 1):
            print(f"  Rank {rank}: Similarity={score:.4f} ({score*100:.2f}%) | Person={person_id}")
else:
    print(f"Query image not found: {query_image}")

## Step 2: Search for Similar Faces
Query with a new photo to find all matching faces in the database with similarity scores

In [ ]:
# Register faces from multiple images
# Replace with your actual image paths

image_paths = [
    r"C:\Users\ASUS\Pictures\New folder\photo_bts_face_recog.jpg",
    # Add more images as needed
]

registered_faces = []

for img_path in image_paths:
    if os.path.exists(img_path):
        person_name = Path(img_path).stem
        print(f"\nProcessing: {person_name}")
        faces = pipeline.detect_and_extract_faces(img_path, person_id=person_name)
        registered_faces.extend(faces)
        print(f"✓ Registered: {person_name} ({len(faces)} face(s))")
    else:
        print(f"File not found: {img_path}")

# Save to database
pipeline.save_database()
print(f"\n✓ Total faces in database: {len(pipeline.faces_db)}")

## Step 1: Register Faces
Upload and register photos to build the face database with automatic multi-face detection

In [ ]:
# Clear old database and initialize fresh InsightFace pipeline
import shutil

# Remove old database if it exists
if os.path.exists("./face_database_insightface"):
    shutil.rmtree("./face_database_insightface")
    print("✓ Old InsightFace database cleared")

# Initialize the pipeline fresh
pipeline = FacialRecognitionPipelineInsightFace(database_dir="./face_database_insightface")
print("\n✓ InsightFace Pipeline ready!")

## Step 0: Initialize InsightFace Pipeline
Initialize the InsightFace-based facial recognition system with superior face detection

In [ ]:
# Import system and InsightFace pipeline
import sys
sys.path.insert(0, r'e:\Face_recog_Detec')

from face_pipeline_insightface import FacialRecognitionPipelineInsightFace
import numpy as np
import pandas as pd
from pathlib import Path
import os
from datetime import datetime
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt
import cv2

print("✓ All imports successful!")

# Facial Recognition Pipeline with InsightFace & Geotag Tracking
## Superior Face Detection, Extraction & Similarity Search System
### InsightFace provides multi-face detection, automatic alignment, and optimized 512D embeddings